# Corporate Bond Valuation - Quantitative Analysis

**Author:** Minjae Seo  
**Purpose:** Computational verification of case study results + sensitivity analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.rcParams.update({'font.size': 12, 'figure.facecolor': 'white'})

## 1. Parameters & Core Functions

In [ ]:
# Bond parameters
F = 100          # Face value
C = 6            # Annual coupon
T = 5            # Maturity (years)
r = 0.04         # Risk-free rate
lam = 0.05       # Hazard rate (default probability)

In [ ]:
def bond_value(R=0, lam=0.05):
    """Calculate bond value given recovery R and hazard rate lambda."""
    pv = 0
    for t in range(1, T+1):
        S_t = (1 - lam) ** t           # Survival prob
        D_t = 1 / (1 + r) ** t         # Discount factor
        P_d = (1 - lam) ** (t-1) * lam # Marginal default prob
        
        pv += C * S_t * D_t            # Expected coupon
        pv += R * P_d * D_t            # Expected recovery
    
    pv += F * (1 - lam) ** T / (1 + r) ** T  # Expected principal
    return pv

def calc_ytm(price):
    """Solve for YTM given price."""
    def pv(y):
        return sum(C / (1+y)**t for t in range(1, T+1)) + F / (1+y)**T - price
    return brentq(pv, 0.001, 0.5)

def calc_spread(price):
    """Credit spread = YTM - risk-free rate."""
    return calc_ytm(price) - r

## 2. Case Study Results

In [ ]:
# Calculate values
V_zero = bond_value(R=0)
V_70 = bond_value(R=70)
spread_zero = calc_spread(V_zero) * 10000
spread_70 = calc_spread(V_70) * 10000

# Results table
results = pd.DataFrame({
    'Part': ['(a)', '(b)', '(c)', '(d)'],
    'Question': ['Bond Value (R=0)', 'Spread (R=0)', 'Bond Value (R=70)', 'Spread (R=70)'],
    'Answer': [f'${V_zero:.2f}', f'{spread_zero:.0f} bps', f'${V_70:.2f}', f'{spread_70:.0f} bps']
})
print(results.to_string(index=False))

## 3. Monte Carlo Verification

In [ ]:
def monte_carlo(R, n=50000):
    """Simulate bond value via Monte Carlo."""
    np.random.seed(42)
    values = []
    
    for _ in range(n):
        pv = 0
        alive = True
        for t in range(1, T+1):
            if alive:
                if np.random.random() < lam:  # Default
                    pv += R / (1+r)**t
                    alive = False
                else:  # Survive
                    pv += C / (1+r)**t
                    if t == T:
                        pv += F / (1+r)**T
        values.append(pv)
    return np.array(values)

mc_zero = monte_carlo(R=0)
mc_70 = monte_carlo(R=70)

print(f"Monte Carlo vs Analytical (n=50,000):")
print(f"  R=$0:  MC=${mc_zero.mean():.2f}, Analytical=${V_zero:.2f}")
print(f"  R=$70: MC=${mc_70.mean():.2f}, Analytical=${V_70:.2f}")

In [ ]:
# Distribution plot
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(mc_zero, bins=40, color='#4A90A4', edgecolor='white', alpha=0.85)
ax[0].axvline(V_zero, color='#C0392B', linestyle='--', linewidth=2, label=f'Analytical: ${V_zero:.2f}')
ax[0].set_xlabel('Bond Value ($)', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].set_title('Zero Recovery', fontsize=14)
ax[0].legend(fontsize=11)

ax[1].hist(mc_70, bins=40, color='#2E7D32', edgecolor='white', alpha=0.85)
ax[1].axvline(V_70, color='#C0392B', linestyle='--', linewidth=2, label=f'Analytical: ${V_70:.2f}')
ax[1].set_xlabel('Bond Value ($)', fontsize=12)
ax[1].set_ylabel('Frequency', fontsize=12)
ax[1].set_title('$70 Recovery', fontsize=14)
ax[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig1_monte_carlo.png', dpi=150)
plt.show()

## 4. Sensitivity Analysis

In [ ]:
# Sensitivity to hazard rate
hazards = np.linspace(0.01, 0.15, 40)
vals_0 = [bond_value(R=0, lam=h) for h in hazards]
vals_70 = [bond_value(R=70, lam=h) for h in hazards]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(hazards*100, vals_0, 'b-', linewidth=2, label='R=$0')
ax[0].plot(hazards*100, vals_70, 'g-', linewidth=2, label='R=$70')
ax[0].axvline(5, color='gray', linestyle=':', linewidth=1.5)
ax[0].scatter([5, 5], [V_zero, V_70], c=['b', 'g'], s=80, zorder=5)
ax[0].set_xlabel('Hazard Rate (%)', fontsize=12)
ax[0].set_ylabel('Bond Value ($)', fontsize=12)
ax[0].legend(fontsize=11)
ax[0].set_title('Value vs Default Probability', fontsize=14)

spreads_0 = [calc_spread(v)*10000 for v in vals_0]
spreads_70 = [calc_spread(v)*10000 for v in vals_70]
ax[1].plot(hazards*100, spreads_0, 'b-', linewidth=2, label='R=$0')
ax[1].plot(hazards*100, spreads_70, 'g-', linewidth=2, label='R=$70')
ax[1].axvline(5, color='gray', linestyle=':', linewidth=1.5)
ax[1].scatter([5, 5], [spread_zero, spread_70], c=['b', 'g'], s=80, zorder=5)
ax[1].set_xlabel('Hazard Rate (%)', fontsize=12)
ax[1].set_ylabel('Credit Spread (bps)', fontsize=12)
ax[1].legend(fontsize=11)
ax[1].set_title('Spread vs Default Probability', fontsize=14)

plt.tight_layout()
plt.savefig('fig2_sensitivity_hazard.png', dpi=150)
plt.show()

In [ ]:
# Sensitivity to recovery
recoveries = np.linspace(0, 100, 40)
vals_r = [bond_value(R=rv) for rv in recoveries]
spreads_r = [calc_spread(v)*10000 for v in vals_r]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(recoveries, vals_r, color='purple', linewidth=2)
ax[0].scatter([0, 70], [V_zero, V_70], c=['b', 'g'], s=80, zorder=5)
ax[0].axhline(100, color='gray', linestyle=':', linewidth=1.5, label='Par')
ax[0].set_xlabel('Recovery Value ($)', fontsize=12)
ax[0].set_ylabel('Bond Value ($)', fontsize=12)
ax[0].set_title('Value vs Recovery', fontsize=14)
ax[0].legend(fontsize=11)

ax[1].plot(recoveries, spreads_r, color='purple', linewidth=2)
ax[1].scatter([0, 70], [spread_zero, spread_70], c=['b', 'g'], s=80, zorder=5)
ax[1].set_xlabel('Recovery Value ($)', fontsize=12)
ax[1].set_ylabel('Credit Spread (bps)', fontsize=12)
ax[1].set_title('Spread vs Recovery', fontsize=14)

plt.tight_layout()
plt.savefig('fig3_sensitivity_recovery.png', dpi=150)
plt.show()

## 5. SQL Demo: Portfolio Analytics

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE bonds (
    bond_id TEXT PRIMARY KEY,
    issuer TEXT,
    coupon REAL,
    rating TEXT
);

CREATE TABLE pricing (
    bond_id TEXT,
    recovery REAL,
    price REAL,
    spread_bps REAL
);

INSERT INTO bonds VALUES 
    ('BOND001', 'Corp A', 6.0, 'BBB'),
    ('BOND002', 'Corp B', 5.5, 'BB');

INSERT INTO pricing VALUES
    ('BOND001', 0, 86.63, 548),
    ('BOND001', 70, 100.79, 181),
    ('BOND002', 40, 92.50, 420);
""")
print("Database created.")

In [ ]:
query = """
SELECT b.issuer, b.rating, p.recovery, p.price, p.spread_bps
FROM bonds b
JOIN pricing p ON b.bond_id = p.bond_id
ORDER BY b.issuer, p.recovery
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))
conn.close()

## Summary

| Metric | R = $0 | R = $70 | Change |
|--------|--------|---------|--------|
| Bond Value | $86.63 | $100.79 | +$14.16 |
| Credit Spread | 548 bps | 181 bps | -367 bps |

**Key insight:** Recovery assumptions dominate credit spread determination. A 70% recovery reduces spread by 67%.